# ERCOT screening analysis — both arms (prompt 26)

16 OAT retrofits (ω = 0.5 wind / 0.4 PV, B = 40) + `__ALL__` + 2 replicate noise-ruler rows,
across `screening_ercot` (v1: availability/contrast arm) and `screening_topup` (actual-top-
curtailer arm). Every read is gated by the replicate-spread noise ruler.

**Footer on every figure: ERCOT: DA≡RT (no forecast error); RUC TimeLimit 120 s (12.1% days
truncated). Never compare cost LEVELS across systems.**

In [1]:
import json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.labelweight": "bold", "axes.titleweight": "bold",
                     "font.weight": "bold", "text.parse_math": False})
from scipy import stats

ERCOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
FOOTER = "ERCOT: DA≡RT (no forecast error); RUC TimeLimit 120 s (12.1% days truncated)"
os.makedirs("figs", exist_ok=True)

def overall(wave_run):
    return pd.read_csv(os.path.join(ERCOT, "extracts", wave_run, "overall.csv")).iloc[0]

def gens(wave_run):
    g = pd.read_csv(os.path.join(ERCOT, "extracts", wave_run, "gen_summary.csv"))
    g["Generator"] = g["Generator"].astype(str)
    g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
    return g

def run_metrics(wave_run):
    o, g = overall(wave_run), gens(wave_run)
    vre = (~g.is_pem) & g.unit_type.isin(["WIND", "PV"])
    return {"true_curt": float(g[vre].curtailment_mwh.sum()),
            "h2": float(g[g.is_pem].curtailment_mwh.sum()),
            "var_cost": float(o["Total generation costs"]),
            "total_cost": float(o["Total costs"]),
            "fixed_cost": float(o["Total fixed costs"]),
            "reserve": float(o["Total reserve shortfall"]),
            "onoffs": float(o["Total on/offs"])}

BASE = run_metrics("base_2019")
gb = gens("base_2019").set_index("Generator")
print("base:", {k: round(v, 1) for k, v in BASE.items()})

RUNS = []
for wave, n in (("screening_ercot", 12), ("screening_topup", 7)):
    dm = pd.read_csv(os.path.join(ERCOT, "waves", wave, "design_matrix.csv"))
    for i in range(1, n + 1):
        d = dm[dm["index"] == i].iloc[0]
        RUNS.append({"wave": wave, "index": i, "site": str(d["oat_site"]),
                     "bus": d["bus"], "type": d["unit_type"], "pmax": d["gen_pmax"],
                     "replicate_of": d.get("replicate_of", np.nan),
                     **run_metrics(f"{wave}/run_index_{i}")})
R = pd.DataFrame(RUNS)
assert len(R) == 19

base: {'true_curt': 9444258.1, 'h2': 0.0, 'var_cost': 1186887019.6, 'total_cost': 7938734507.3, 'fixed_cost': 6751847487.7, 'reserve': 280.9, 'onoffs': 17689.0}


/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_88687/648128296.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_88687/648128296.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_88687/648128296.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future 

## Noise ruler (2 replicate pairs) — and the ±$79M total-cost trap

`__BASE_REPLICATE__` (row 11) repeats the base; row 12 repeats row 1 (site 275). The ruler for
each objective = the larger of the two |replicate − original| spreads. **Total cost is excluded
as an objective**: the fixed-cost component is bimodal — the runs split into two commitment
states ≈ $79M apart, and per-generator attribution puts essentially the whole gap on
**generator 1, the 2,430 MW nuclear** (25 ↔ 26 starts, ±104 GWh output, ±$79.6M Unit Cost).
Under TimeLimit=120 the solver toggles that state nondeterministically, so total-cost deltas at
screening scale are unreadable. Variable generation cost is tight and is the cost read-out.

In [2]:
rep_b = R[(R.wave == "screening_ercot") & (R["index"] == 11)].iloc[0]
r1 = R[(R.wave == "screening_ercot") & (R["index"] == 1)].iloc[0]
r12 = R[(R.wave == "screening_ercot") & (R["index"] == 12)].iloc[0]
METRICS = ["true_curt", "h2", "var_cost", "reserve", "onoffs", "total_cost", "fixed_cost"]
ruler = {}
for m in METRICS:
    d1, d2 = abs(rep_b[m] - BASE[m]), abs(r12[m] - r1[m])
    ruler[m] = max(d1, d2)
    print(f"{m:11s} |rep-base|={d1:14,.1f}  |r12-r1|={d2:14,.1f}  RULER={ruler[m]:14,.1f}")
print(f"\nfixed-cost states: base {BASE['fixed_cost']/1e6:,.1f}M / rep {rep_b['fixed_cost']/1e6:,.1f}M ; "
      f"r1 {r1['fixed_cost']/1e6:,.1f}M / r12 {r12['fixed_cost']/1e6:,.1f}M  -> bimodal ±$79M (gen 1 = 2,430 MW NUC)")
json.dump({k: float(v) for k, v in ruler.items()}, open("screening_noise_ruler.json", "w"), indent=2)

true_curt   |rep-base|=       5,479.1  |r12-r1|=       5,007.0  RULER=       5,479.1
h2          |rep-base|=           0.0  |r12-r1|=       2,914.1  RULER=       2,914.1
var_cost    |rep-base|=     285,620.6  |r12-r1|=     228,430.6  RULER=     285,620.6
reserve     |rep-base|=         173.8  |r12-r1|=         554.2  RULER=         554.2
onoffs      |rep-base|=          46.0  |r12-r1|=         258.0  RULER=         258.0
total_cost  |rep-base|=  79,451,497.0  |r12-r1|=  79,275,607.5  RULER=  79,451,497.0
fixed_cost  |rep-base|=  79,165,876.4  |r12-r1|=  79,504,038.2  RULER=  79,504,038.2

fixed-cost states: base 6,751.8M / rep 6,831.0M ; r1 6,825.4M / r12 6,745.9M  -> bimodal ±$79M (gen 1 = 2,430 MW NUC)


## Site effects, both arms — gated deltas

Relief = −Δ(true curtailment). The Prescient artifact is handled: `overall.csv` counts the PEM
twin's H2 as "curtailment"; true curtailment is `gen_summary` non-PEM WIND+PV only (hydro
excluded throughout).

In [3]:
oat = R[~R.site.isin(["__ALL__", "__BASE_REPLICATE__"]) & R.replicate_of.isna()].copy()
oat["d_true_curt"] = oat.true_curt - BASE["true_curt"]
oat["relief"] = -oat.d_true_curt
oat["d_var_cost"] = oat.var_cost - BASE["var_cost"]
oat["d_reserve"] = oat.reserve - BASE["reserve"]
oat["d_onoffs"] = oat.onoffs - BASE["onoffs"]
# base per-site curtailment + availability from the base gen_summary
oat["site_base_curt"] = [float(gb.loc[s, "curtailment_mwh"]) for s in oat.site]
oat["site_avail"] = [float(gb.loc[s, "curtailment_mwh"] + gb.loc[s, "output_mwh"]) for s in oat.site]
oat["relief_per_h2"] = oat.relief / oat.h2
oat["own_share"] = oat.site_base_curt / BASE["true_curt"]
for m, r in (("relief", ruler["true_curt"]), ("d_var_cost", ruler["var_cost"]),
             ("d_reserve", ruler["reserve"]), ("d_onoffs", ruler["onoffs"])):
    oat[f"{m}_x_ruler"] = oat[m] / r
show = oat[["wave", "site", "bus", "type", "pmax", "h2", "relief", "relief_x_ruler",
            "relief_per_h2", "site_base_curt", "d_var_cost", "d_var_cost_x_ruler",
            "d_reserve_x_ruler", "d_onoffs_x_ruler"]].sort_values("relief", ascending=False)
with pd.option_context("display.width", 220):
    print(show.round(2).to_string(index=False))
oat.to_csv("screening_site_effects.csv", index=False)
print(f"\nrulers: curt {ruler['true_curt']:,.0f} MWh, var-cost ${ruler['var_cost']/1e6:.2f}M, "
      f"reserve {ruler['reserve']:,.0f} MWh, on/offs {ruler['onoffs']:.0f}")

           wave site   bus type    pmax         h2     relief  relief_x_ruler  relief_per_h2  site_base_curt  d_var_cost  d_var_cost_x_ruler  d_reserve_x_ruler  d_onoffs_x_ruler
screening_ercot  275 120.0 WIND 1039.00 3017560.71 2172937.63          396.59           0.72      1973765.11  6243473.68               21.86               0.41             -2.52
screening_ercot  274 120.0 WIND  746.40 2172394.14 1568002.48          286.18           0.72      1521527.95  4377401.61               15.33               1.23             -0.70
screening_topup  270 120.0 WIND  579.00 1687647.42 1321264.97          241.15           0.78      1319785.15  2309065.72                8.08               0.88             -0.20
screening_topup  110   2.0 WIND  338.60 1066036.26  686606.34          125.31           0.64       344842.69  3478253.28               12.18              -0.31             -1.53
screening_ercot   30  26.0   PV  926.67 1294478.97  674572.52          123.12           0.52       228404.18  

## What does retrofit value track — curtailment, or size/availability?

The two-arm design's question. Rank correlations of system relief against the site's own
base-case curtailment vs against its available energy; bus-120 (the congestion pocket hosting
5 of the top-8 curtailers) highlighted.

In [4]:
for xvar in ("site_base_curt", "site_avail", "pmax", "h2"):
    sp = stats.spearmanr(oat[xvar], oat.relief)
    kt = stats.kendalltau(oat[xvar], oat.relief)
    print(f"relief vs {xvar:15s}: Spearman {sp.statistic:+.3f} (p={sp.pvalue:.3g}), "
          f"Kendall {kt.statistic:+.3f}")
b120 = oat[oat.bus == 120.0]; rest = oat[oat.bus != 120.0]
print(f"\nbus-120 sites (n={len(b120)}): median relief/h2 = {b120.relief_per_h2.median():.2f}, "
      f"median relief/pmax = {(b120.relief/b120.pmax).median():,.0f} MWh/MW")
print(f"other sites (n={len(rest)}): median relief/h2 = {rest.relief_per_h2.median():.2f}, "
      f"median relief/pmax = {(rest.relief/rest.pmax).median():,.0f} MWh/MW")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, xvar, xl in ((axes[0], "site_base_curt", "site's own base-case curtailment [MWh]"),
                     (axes[1], "site_avail", "site's annual available energy [MWh]")):
    for m, lab, c in ((oat.bus == 120.0, "bus 120 (congestion pocket)", "crimson"),
                      (oat.bus != 120.0, "other buses", "tab:blue")):
        sub = oat[m]
        mk = {"WIND": "o", "PV": "s"}
        for t in ("WIND", "PV"):
            ss = sub[sub.type == t]
            ax.scatter(ss[xvar], ss.relief, c=c, marker=mk[t], s=55,
                       label=f"{lab} ({t})" if len(ss) else None)
        for _, rr in sub.iterrows():
            ax.annotate(rr.site, (rr[xvar], rr.relief), fontsize=7,
                        xytext=(3, 3), textcoords="offset points")
    ax.axhline(ruler["true_curt"], color="k", ls=":", lw=1)
    ax.axhline(2 * ruler["true_curt"], color="k", ls="--", lw=1)
    ax.set_xlabel(xl); ax.set_ylabel("system curtailment relief [MWh]")
sp1 = stats.spearmanr(oat.site_base_curt, oat.relief).statistic
sp2 = stats.spearmanr(oat.site_avail, oat.relief).statistic
axes[0].set_title(f"vs own curtailment (Spearman {sp1:+.2f})")
axes[1].set_title(f"vs available energy (Spearman {sp2:+.2f})")
axes[0].legend(fontsize=7)
fig.suptitle("What does retrofit value track? (dotted/dashed = 1x/2x noise ruler)")
fig.text(0.5, 0.005, FOOTER, ha="center", fontsize=7, style="italic")
fig.tight_layout(rect=[0, 0.02, 1, 1])
fig.savefig("figs/screening_value_tracking.png", dpi=150); plt.close(fig)
print("figure written")

relief vs site_base_curt : Spearman +0.888 (p=4.33e-06), Kendall +0.767
relief vs site_avail     : Spearman -0.029 (p=0.914), Kendall +0.033
relief vs pmax           : Spearman -0.171 (p=0.528), Kendall -0.067
relief vs h2             : Spearman -0.041 (p=0.88), Kendall +0.033

bus-120 sites (n=5): median relief/h2 = 0.72, median relief/pmax = 2,101 MWh/MW
other sites (n=11): median relief/h2 = 0.37, median relief/pmax = 495 MWh/MW


figure written


## Σ(OAT) vs `__ALL__` — interaction check (v1 arm's 9 sites)

In [5]:
allrow = R[R.site == "__ALL__"].iloc[0]
v1 = oat[oat.wave == "screening_ercot"]
for m, r in (("relief", ruler["true_curt"]), ("d_var_cost", ruler["var_cost"])):
    sig = v1[m].sum()
    joint = (BASE["true_curt"] - allrow.true_curt) if m == "relief" else (allrow.var_cost - BASE["var_cost"])
    print(f"{m:10s}: Sigma(OAT 9 sites) = {sig:,.0f}   joint __ALL__ = {joint:,.0f}   "
          f"joint/Sigma = {joint/sig:.3f}   (Sigma-joint)/ruler = {(sig-joint)/r:,.1f}x")
print(f"__ALL__ h2 = {allrow.h2:,.0f} MWh vs Sigma h2 = {v1.h2.sum():,.0f}")

relief    : Sigma(OAT 9 sites) = 5,422,221   joint __ALL__ = 5,059,920   joint/Sigma = 0.933   (Sigma-joint)/ruler = 66.1x
d_var_cost: Sigma(OAT 9 sites) = 81,115,371   joint __ALL__ = 74,512,756   joint/Sigma = 0.919   (Sigma-joint)/ruler = 23.1x
__ALL__ h2 = 18,894,639 MWh vs Sigma h2 = 18,946,605


In [6]:
summary = {
    "ruler": {k: float(v) for k, v in ruler.items()},
    "bimodal_fixed_cost": {"gap_musd": 79.5, "unit": "gen 1 (2,430 MW NUC)",
                           "evidence": "25 vs 26 starts, +104 GWh, +$79.6M Unit Cost"},
    "site_effects": oat[["wave", "site", "bus", "type", "pmax", "h2", "relief",
                         "relief_x_ruler", "relief_per_h2", "site_base_curt",
                         "d_var_cost", "d_var_cost_x_ruler"]].to_dict("records"),
    "rank_correlations": {x: float(stats.spearmanr(oat[x], oat.relief).statistic)
                          for x in ("site_base_curt", "site_avail", "pmax", "h2")},
    "bus120_median_relief_per_h2": float(b120.relief_per_h2.median()),
    "others_median_relief_per_h2": float(rest.relief_per_h2.median()),
    "sigma_vs_all": {"relief_ratio": float((BASE["true_curt"] - allrow.true_curt) / v1.relief.sum())},
}
json.dump(summary, open("screening_summary.json", "w"), indent=2, default=float)
print("screening_summary.json written")

screening_summary.json written
